# Classifying Particles with a Convolutional Neural Network


<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<a href="https://colab.research.google.com/github/Leolinen/TIF360/blob/main/HW2/particles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<strong>If using Colab/Kaggle:</strong> You need to uncomment the code in the cell below this one.
</div>

In [147]:
!pip install deeplay  # Uncomment if using Colab/Kaggle.

In [148]:
import pickle
import torch
import torch.utils.data
import matplotlib.pyplot as plt
import numpy as np
import deeplay as dl
import torchmetrics as tm
from torch.nn.functional import relu
from skimage.exposure import rescale_intensity
from skimage.transform import resize


## Loading the Simple Particle Dataset


In [149]:
with open("simple_particle_dataset.pkl", "rb") as f:
    raw = pickle.load(f)


class ParticleDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        self.images = [
            torch.tensor(img, dtype=torch.float32).unsqueeze(0)
            for img in images
        ]
        self.labels = [
            torch.tensor(float(lbl[0]), dtype=torch.float32).unsqueeze(-1)
            for lbl in labels
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


dataset = ParticleDataset(raw["images"], raw["labels"])


### Splitting the Dataset and Defining the Data Loaders

In [150]:
train, test = torch.utils.data.random_split(dataset, [0.8, 0.2])

... and define the data loaders.

In [151]:
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=256, shuffle=False)

### Plotting the ROC Curve

Implement a function to plot the ROC curve ...

In [152]:
def plot_roc(classifier, loader):
    """Plot ROC curve."""
    roc = tm.ROC(task="binary")
    f1 = tm.F1Score(task="binary")
    for image, label in loader:
        preds = classifier(image)
        roc.update(preds, label.long())
        f1.update(preds, label.long())

    fig, ax = roc.plot(score=True)
    ax.grid(False)
    ax.axis("square")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="center right")
    plt.show()

    print(f"F1 Score: {f1.compute():.4f}")


## Classifying the Particles with Convolutional Neural Networks

Implement a convolutional neural network with a dense top ...

In [153]:
conv_base = dl.ConvolutionalNeuralNetwork(
    in_channels=1, hidden_channels=[16, 16], out_channels=16,
)
conv_base.blocks[2].pool.configure(torch.nn.MaxPool2d, kernel_size=4)

connector = dl.Layer(torch.nn.AdaptiveAvgPool2d, output_size=1)

dense_top = dl.MultiLayerPerceptron(
    in_features=16, hidden_features=[], out_features=1,
    out_activation=torch.nn.Sigmoid,
)

cnn = dl.Sequential(conv_base, connector, dense_top)


In [154]:
cnn_classifier = dl.BinaryClassifier(
    model=cnn, optimizer=dl.RMSprop(lr=0.0001),
).create()

### Training the Convolutional Neural Network

In [155]:
cnn_trainer = dl.Trainer(max_epochs=5, accelerator="auto")
cnn_trainer.fit(cnn_classifier, train_loader)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Testing the Convolutional Neural Network

In [ ]:
cnn_trainer.test(cnn_classifier, test_loader)

### Plotting the ROC Curve

In [ ]:
plot_roc(cnn_classifier, test_loader)

## Hard Particle Dataset

In [ ]:
with open("hard_particle_dataset.pkl", "rb") as f:
    raw = pickle.load(f)


class ParticleDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        self.images = [
            torch.tensor(img, dtype=torch.float32).unsqueeze(0)
            for img in images
        ]
        self.labels = [
            torch.tensor(float(lbl[0]), dtype=torch.float32).unsqueeze(-1)
            for lbl in labels
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


dataset = ParticleDataset(raw["images"], raw["labels"])


### Splitting the Dataset and Defining the Data Loaders

In [ ]:
train, test = torch.utils.data.random_split(dataset, [0.8, 0.2])

... and define the data loaders.

In [ ]:
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=256, shuffle=False)

## Classifying the Particles with Convolutional Neural Networks

Implement a convolutional neural network with a dense top ...

In [ ]:
conv_base = dl.ConvolutionalNeuralNetwork(
    in_channels=1, hidden_channels=[16, 16], out_channels=16,
)
conv_base.blocks[2].pool.configure(torch.nn.MaxPool2d, kernel_size=10)

connector = dl.Layer(torch.nn.AdaptiveAvgPool2d, output_size=1)

dense_top = dl.MultiLayerPerceptron(
    in_features=16, hidden_features=[], out_features=6,
    out_activation=torch.nn.Softmax,
)

cnn = dl.Sequential(conv_base, connector, dense_top)


In [ ]:
cnn_classifier = dl.Classifier(
    model=cnn, optimizer=dl.RMSprop(lr=0.001),
).create()


### Training the Convolutional Neural Network

In [ ]:
cnn_trainer = dl.Trainer(max_epochs=5, accelerator="auto")
cnn_trainer.fit(cnn_classifier, train_loader)

### Testing the Convolutional Neural Network

In [ ]:
cnn_trainer.test(cnn_classifier, test_loader)

### Plotting the ROC Curve

In [ ]:
plot_roc(cnn_classifier, test_loader)